In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# common daily business day index
IDX = pd.bdate_range('1999-01-04', '2024-12-31')
print(f"Index: {IDX[0].date()} -> {IDX[-1].date()} ({len(IDX):,} days)")

# raw series storage — all fetch cells write here
raw = {}

Index: 1999-01-04 -> 2024-12-31 (6,782 days)


In [3]:
import yfinance as yf
import time
import warnings
warnings.filterwarnings('ignore')

START = '1995-01-01'

def fetch_yf(name, ticker, delay=2):
    time.sleep(delay)
    try:
        d = yf.download(ticker, start=START, progress=False, auto_adjust=True)
        d = d.dropna(how='all')
        if len(d) > 0:
            raw[name] = d['Close'].squeeze()        # ← added
            print(f"{name:10s} {ticker:15s}: {d.index[0].date()} -> {d.index[-1].date()} ({len(d):,} rows)")
        else:
            print(f"{name:10s} {ticker:15s}: NO DATA")
    except Exception as e:
        print(f"{name:10s} {ticker:15s}: ERROR")

print('\n=== EQUITY INDICES ===')
fetch_yf('SPX',   '^GSPC')
fetch_yf('STOXX', '^STOXX50E')
fetch_yf('FTSE',  '^FTSE')
fetch_yf('N225',  '^N225')

print('\n=== VOL INDICES ===')
fetch_yf('VIX_YF', '^VIX')
fetch_yf('MOVE',   '^MOVE')

print('\n=== COMMODITIES ===')
fetch_yf('GOLD',   'GC=F')

print(f"\nraw keys: {list(raw.keys())}")


=== EQUITY INDICES ===
SPX        ^GSPC          : 1995-01-03 -> 2026-05-20 (7,898 rows)
STOXX      ^STOXX50E      : 2007-03-30 -> 2026-05-20 (4,795 rows)
FTSE       ^FTSE          : 1995-01-03 -> 2026-05-20 (7,926 rows)
N225       ^N225          : 1995-01-04 -> 2026-05-21 (7,694 rows)

=== VOL INDICES ===
VIX_YF     ^VIX           : 1995-01-03 -> 2026-05-20 (7,898 rows)
MOVE       ^MOVE          : 2002-11-12 -> 2026-05-20 (5,815 rows)

=== COMMODITIES ===
GOLD       GC=F           : 2000-08-30 -> 2026-05-20 (6,454 rows)

raw keys: ['SPX', 'STOXX', 'FTSE', 'N225', 'VIX_YF', 'MOVE', 'GOLD']


In [4]:
from dotenv import load_dotenv
from pathlib import Path
import os, requests, time
import pandas as pd

load_dotenv(dotenv_path=Path('../.env'))
FRED_API_KEY = os.getenv('FRED_API_KEY')

def fetch_fred(name, series, delay=1):
    time.sleep(delay)
    try:
        url = (
            f'https://api.stlouisfed.org/fred/series/observations'
            f'?series_id={series}'
            f'&api_key={FRED_API_KEY}'
            f'&file_type=json'
            f'&observation_start=1990-01-01'
            f'&limit=100000'
            f'&sort_order=asc'
        )
        r = requests.get(url, timeout=15)
        data = r.json()
        if 'error_code' in data:
            print(f"{name:20s} {series:25s}: API ERROR — {data.get('error_message')}")
            return
        obs = [o for o in data['observations'] if o['value'] != '.']
        if obs:
            first = obs[0]['date']
            last  = obs[-1]['date']
            raw[name] = pd.Series(                  # ← added
                {pd.Timestamp(o['date']): float(o['value']) for o in obs}
            )
            print(f"{name:20s} {series:25s}: {first} -> {last} ({len(obs):,} rows)")
        else:
            print(f"{name:20s} {series:25s}: NO DATA")
    except Exception as e:
        print(f"{name:20s} {series:25s}: ERROR — {str(e)[:40]}")

print('=== TARGET PAIRS ===')
fetch_fred('EURUSD',    'DEXUSEU')
fetch_fred('GBPUSD',    'DEXUSUK')
fetch_fred('JPYUSD',    'DEXJPUS')

print('\n=== POLICY RATES ===')
fetch_fred('FED_FUNDS', 'FEDFUNDS')

print('\n=== US YIELDS ===')
fetch_fred('US_10Y',    'DGS10')
fetch_fred('US_2Y',     'DGS2')
fetch_fred('DTB3',      'DTB3')

print('\n=== FOREIGN YIELDS MONTHLY ===')
fetch_fred('BUND_M',    'IRLTLT01DEM156N')
fetch_fred('GILT_M',    'IRLTLT01GBM156N')
fetch_fred('BTP_M',     'IRLTLT01ITM156N')
fetch_fred('JGB_M',     'IRLTLT01JPM156N')

print('\n=== VOL ===')
fetch_fred('VIX',       'VIXCLS')
fetch_fred('EVZ',       'EVZCLS')

print('\n=== FUNDING STRESS ===')
fetch_fred('TED',       'TEDRATE')
fetch_fred('SOFR',      'SOFR')

print('\n=== OIL ===')
fetch_fred('WTI',       'DCOILWTICO')

print(f"\nraw keys: {list(raw.keys())}")

=== TARGET PAIRS ===
EURUSD               DEXUSEU                  : 1999-01-04 -> 2026-05-15 (6,864 rows)
GBPUSD               DEXUSUK                  : 1990-01-02 -> 2026-05-15 (9,128 rows)
JPYUSD               DEXJPUS                  : 1990-01-02 -> 2026-05-15 (9,128 rows)

=== POLICY RATES ===
FED_FUNDS            FEDFUNDS                 : 1990-01-01 -> 2026-04-01 (436 rows)

=== US YIELDS ===
US_10Y               DGS10                    : 1990-01-02 -> 2026-05-19 (9,101 rows)
US_2Y                DGS2                     : 1990-01-02 -> 2026-05-19 (9,101 rows)
DTB3                 DTB3                     : 1990-01-02 -> 2026-05-19 (9,101 rows)

=== FOREIGN YIELDS MONTHLY ===
BUND_M               IRLTLT01DEM156N          : 1990-01-01 -> 2026-04-01 (436 rows)
GILT_M               IRLTLT01GBM156N          : 1990-01-01 -> 2026-04-01 (436 rows)
BTP_M                IRLTLT01ITM156N          : 1991-03-01 -> 2026-03-01 (421 rows)
JGB_M                IRLTLT01JPM156N          : 1990-0

In [5]:
from ecbdata import ecbdata as ecb
import requests, time
import pandas as pd
import numpy as np

print('=== ECB via ecbdata package ===')

def fetch_ecb_pkg(name, series_key):
    try:
        df = ecb.get_series(series_key, start='1999-01')
        df = df[df['OBS_VALUE'].notna()].copy()
        if len(df) > 0:
            raw[name] = pd.Series(                  # ← added
                df['OBS_VALUE'].values,
                index=pd.to_datetime(df['TIME_PERIOD'])
            )
            print(f"{name:20s} {series_key:45s}: "
                  f"{df['TIME_PERIOD'].iloc[0]} -> "
                  f"{df['TIME_PERIOD'].iloc[-1]} "
                  f"({len(df):,} rows)")
        else:
            print(f"{name:20s}: NO DATA")
    except Exception as e:
        print(f"{name:20s}: ERROR — {e}")

fetch_ecb_pkg('ECB_RATE',   'FM.B.U2.EUR.4F.KR.DFR.LEV')
fetch_ecb_pkg('BUND_DAILY', 'YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y')

print()
print('=== BoE API ===')

def fetch_boe(name, series, delay=2):
    time.sleep(delay)
    try:
        url = (
            'https://www.bankofengland.co.uk/boeapps/database/'
            '_iadb-FromShowColumns.asp'
            f'?csv.x=yes&Datefrom=01/Jan/1975&Dateto=now'
            f'&SeriesCodes={series}&CSVF=TT&UsingCodes=Y'
        )
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        r = requests.get(url, headers=headers, timeout=15)
        lines = r.text.strip().split('\n')
        data_lines = [l for l in lines
                      if l and l[0].isdigit() and ',' in l]
        if data_lines:
            dates = [pd.Timestamp(l.split(',')[0]) for l in data_lines]
            vals  = [float(l.split(',')[1])
                     if l.split(',')[1].strip() else np.nan
                     for l in data_lines]
            s = pd.Series(vals, index=dates).dropna()
            raw[name] = s                           # ← added
            first = data_lines[0].split(',')[0]
            last  = data_lines[-1].split(',')[0]
            print(f"{name:20s} {series:15s}: "
                  f"{first} -> {last} ({len(data_lines):,} rows)")
        else:
            print(f"{name:20s}: NO DATA")
    except Exception as e:
        print(f"{name:20s}: ERROR — {e}")

fetch_boe('BOE_RATE',  'IUMABEDR')
fetch_boe('GILT_SPOT', 'IUDMNPY')

print(f"\nTotal raw series stored: {len(raw)}")
print(f"Keys: {list(raw.keys())}")

=== ECB via ecbdata package ===
ECB_RATE             FM.B.U2.EUR.4F.KR.DFR.LEV                    : 1999-01-01 -> 2025-06-11 (67 rows)
BUND_DAILY           YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y          : 2004-09-06 -> 2026-05-19 (5,546 rows)

=== BoE API ===
BOE_RATE             IUMABEDR       : 31 Jan 1975 -> 30 Apr 2026 (616 rows)
GILT_SPOT            IUDMNPY        : 01 Nov 1993 -> 18 May 2026 (8,222 rows)

Total raw series stored: 27
Keys: ['SPX', 'STOXX', 'FTSE', 'N225', 'VIX_YF', 'MOVE', 'GOLD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'FED_FUNDS', 'US_10Y', 'US_2Y', 'DTB3', 'BUND_M', 'GILT_M', 'BTP_M', 'JGB_M', 'VIX', 'EVZ', 'TED', 'SOFR', 'WTI', 'ECB_RATE', 'BUND_DAILY', 'BOE_RATE', 'GILT_SPOT']


In [6]:
# create master index 

eurusd_raw = raw['EURUSD'].dropna()
eurusd_raw.index = pd.to_datetime(eurusd_raw.index)
IDX = eurusd_raw.index[
    (eurusd_raw.index >= pd.Timestamp('1999-01-04')) &
    (eurusd_raw.index <= pd.Timestamp('2024-12-31'))
]
print(f"FX trading days (master index): {len(IDX):,}")
print(f"Synthetic bdate_range would be: "
      f"{len(pd.bdate_range('1999-01-04', '2024-12-31')):,}")
print(f"US holidays correctly removed:  "
      f"{len(pd.bdate_range('1999-01-04', '2024-12-31')) - len(IDX):,}")

# normalize asian date time

ASIAN_NODES = ['N225', 'JGB_M']
raw_adjusted = {}
for name, s in raw.items():
    s_adj = s.copy()
    s_adj.index = pd.to_datetime(s_adj.index)
    if name in ASIAN_NODES:
        s_adj.index = s_adj.index + pd.offsets.BDay(1)
        print(f"NY-adjusted {name}: lagged 1 business day")
    raw_adjusted[name] = s_adj

# isolate daily vs monthly nodes, different forward filling across nans

MONTHLY_NODES = [
    'FED_FUNDS', 'ECB_RATE', 'BOE_RATE',
    'BUND_M', 'GILT_M', 'BTP_M', 'JGB_M'
]

# mask unavail data
nan_mask_pre = pd.DataFrame(False, index=IDX,
                             columns=list(raw_adjusted.keys()))

aligned = {}
for name, s in raw_adjusted.items():
    s_clean = s.copy()
    s_clean = s_clean[~s_clean.index.duplicated(keep='last')]
    s_reindexed = s_clean.reindex(IDX)

    if name in MONTHLY_NODES:
        # rate holds between meetings — only mask before series starts
        first_valid = s_reindexed.first_valid_index()
        nan_mask_pre[name] = (IDX < first_valid) if first_valid is not None else True
        aligned[name] = s_reindexed.ffill()
    else:
        # daily series — mask any day without real observation
        nan_mask_pre[name] = s_reindexed.isna()
        aligned[name] = s_reindexed.ffill(limit=1)

# adjust directionality of USD/JPY
aligned['USDJPY'] = 1 / aligned['JPYUSD']
nan_mask_pre['USDJPY'] = nan_mask_pre['JPYUSD']

# inspection
print(f"\nAll series standardized to NY close (5pm EST)")
print(f"Master index: {IDX[0].date()} -> {IDX[-1].date()} "
      f"({len(IDX):,} FX trading days)\n")

print("Masked days per node (pre-fill — what graph transformer will see):")
any_masked = False
for name in sorted(nan_mask_pre.columns):
    n = nan_mask_pre[name].sum()
    if n > 0:
        any_masked = True
        first_valid = aligned[name].first_valid_index()
        pct = 100 * n / len(IDX)
        print(f"  {name:20s}: {n:,} days masked ({pct:.1f}%) — "
              f"first valid: "
              f"{first_valid.date() if first_valid else 'NONE'}")

if not any_masked:
    print("  None — all series fully available")

print(f"\nRemaining NaNs after fill (unfillable gaps):")
any_nan = False
for name in sorted(aligned.keys()):
    n = aligned[name].isna().sum()
    if n > 0:
        any_nan = True
        pct = 100 * n / len(IDX)
        print(f"  {name:20s}: {n:,} NaNs ({pct:.1f}%)")

if not any_nan:
    print("  None — all gaps filled within limit")

print(f"\nTotal aligned series: {len(aligned)}")
print(f"nan_mask shape: {nan_mask_pre.shape}")

FX trading days (master index): 6,520
Synthetic bdate_range would be: 6,782
US holidays correctly removed:  262
NY-adjusted N225: lagged 1 business day
NY-adjusted JGB_M: lagged 1 business day

All series standardized to NY close (5pm EST)
Master index: 1999-01-04 -> 2024-12-31 (6,520 FX trading days)

Masked days per node (pre-fill — what graph transformer will see):
  BOE_RATE            : 60 days masked (0.9%) — first valid: 1999-03-31
  BTP_M               : 19 days masked (0.3%) — first valid: 1999-02-01
  BUND_DAILY          : 1,500 days masked (23.0%) — first valid: 2004-09-07
  BUND_M              : 19 days masked (0.3%) — first valid: 1999-02-01
  DTB3                : 28 days masked (0.4%) — first valid: 1999-01-04
  EVZ                 : 2,252 days masked (34.5%) — first valid: 2007-11-01
  FED_FUNDS           : 19 days masked (0.3%) — first valid: 1999-02-01
  FTSE                : 148 days masked (2.3%) — first valid: 1999-01-04
  GILT_M              : 19 days masked (0.3%

In [8]:
# Cell 6 — Derived Nodes

# bund: merge FRED monthly pre-2004, ECB daily post-2004
bund_cutoff = pd.Timestamp('2004-09-06')
bund = aligned['BUND_M'].copy()
bund.loc[bund_cutoff:] = aligned['BUND_DAILY'].loc[bund_cutoff:]
aligned['BUND'] = bund
nan_mask_pre['BUND'] = nan_mask_pre['BUND_M'].copy()
nan_mask_pre['BUND'].loc[bund_cutoff:] = nan_mask_pre['BUND_DAILY'].loc[bund_cutoff:]

# rate differentials
aligned['US_EU_2Y'] = aligned['US_2Y']    - aligned['BUND']
aligned['UK_EU_2Y'] = aligned['GILT_SPOT'] - aligned['BUND']
aligned['BTP_BUND'] = aligned['BTP_M']    - aligned['BUND']

nan_mask_pre['US_EU_2Y'] = nan_mask_pre['US_2Y']     | nan_mask_pre['BUND']
nan_mask_pre['UK_EU_2Y'] = nan_mask_pre['GILT_SPOT'] | nan_mask_pre['BUND']
nan_mask_pre['BTP_BUND'] = nan_mask_pre['BTP_M']     | nan_mask_pre['BUND']

# funding stress: TED through 2022, SOFR-DTB3 after
ted_end    = pd.Timestamp('2022-01-21')
sofr_start = pd.Timestamp('2022-01-24')
ted_sofr   = aligned['TED'].copy()
ted_sofr.loc[sofr_start:] = (aligned['SOFR'] - aligned['DTB3']).loc[sofr_start:]
aligned['TED_SOFR'] = ted_sofr
nan_mask_pre['TED_SOFR'] = nan_mask_pre['TED'].copy()
nan_mask_pre['TED_SOFR'].loc[sofr_start:] = nan_mask_pre['SOFR'].loc[sofr_start:]

# report
for name in ['BUND', 'US_EU_2Y', 'UK_EU_2Y', 'BTP_BUND', 'TED_SOFR']:
    s    = aligned[name]
    mask = nan_mask_pre[name]
    print(f"{name:15s}: mean={s.mean():.3f}  std={s.std():.3f}  "
          f"masked={mask.sum():,} ({100*mask.sum()/len(IDX):.1f}%)")

BUND           : mean=2.540  std=1.801  masked=91 (1.4%)
US_EU_2Y       : mean=-0.219  std=1.710  masked=105 (1.6%)
UK_EU_2Y       : mean=0.699  std=0.550  masked=185 (2.8%)
BTP_BUND       : mean=1.144  std=0.935  masked=91 (1.4%)
TED_SOFR       : mean=0.365  std=0.421  masked=141 (2.2%)


In [ ]:
# Cell 7 — Final Node List, Feature Computation, Output

# ── final node ordering — target pairs MUST be indices 0, 1, 2 ────────
NODE_NAMES = [
    # target pairs — fixed indices 0, 1, 2
    'EURUSD', 'GBPUSD', 'USDJPY',
    # policy rates
    'FED_FUNDS', 'ECB_RATE', 'BOE_RATE',
    # government yields
    'US_10Y', 'US_2Y', 'BUND', 'GILT_SPOT', 'BTP_M', 'JGB_M',
    # rate differentials
    'US_EU_2Y', 'UK_EU_2Y', 'BTP_BUND',
    # equity indices
    'SPX', 'STOXX', 'FTSE', 'N225',
    # vol
    'VIX', 'EVZ', 'MOVE',
    # funding stress
    'TED_SOFR',
    # commodities
    'GOLD', 'WTI',
]

print(f"Total nodes: {len(NODE_NAMES)}")
print(f"input_dim:   {len(NODE_NAMES) * 3}")
print(f"Target pairs at indices 0,1,2: {NODE_NAMES[:3]}")

# confirm all nodes in presence
missing = [n for n in NODE_NAMES if n not in aligned]
if missing:
    print(f"WARNING — missing from aligned: {missing}")
else:
    print("All nodes confirmed in aligned")

Total nodes: 25
input_dim:   75
Target pairs at indices 0,1,2: ['EURUSD', 'GBPUSD', 'USDJPY']
All nodes confirmed in aligned


In [11]:
print("In NODE_NAMES but not in aligned:")
for n in NODE_NAMES:
    if n not in aligned:
        print(f"  {n}")

In NODE_NAMES but not in aligned:
